# 03 — Modelli a grafo: DGCNN e DHSLP
- **DGCNN**: **grafo PCC (o PLV) per-trial + pruning top-k** (tecnica base della tesi), node features
  dal **segnale grezzo** (encoder temporale), non band-power. 1 grafo per trial (regola CLAUDE.md).
- **DHSLP**: **fedele a EEG_13b** — iperarchi = embedding **apprendibili**, incidenza soft
  `H = softmax(node·E)`, K finestre raw + positional encoding, HGNN conv (Feng 2019).

Entrambi usano `PP_MINIMAL` (default) e girano su tutti i protocolli
(`run_subject_dependent` / `run_subject_mixed` / `run_subject_independent`).

In [ ]:
# --- setup: rende importabili i moduli track3_*.py ---
import sys, os
sys.path.insert(0, os.path.abspath('.'))
import numpy as np, matplotlib.pyplot as plt
import track3_config as C, track3_io as io, track3_preproc as P
print(C.summary())
assert C.DATA_ROOT is not None, C._no_data_msg()
import track3_train as T
import track3_models as M
device=C.get_device(); print('device:', device)

## 1. DGCNN — grafo PCC per-trial + pruning (node features dal raw)

In [ ]:
df_dg, res_dg = T.run_subject_dependent(
    'dgcnn', model_kwargs=dict(metric='pcc', k_neighbors=8, emb_dim=64, hid=64),
    train_kwargs=dict(epochs=200, patience=30, lr=1e-3, batch_size=32))
T.save_metrics(df_dg, 'dgcnn')
T.plot_per_subject(df_dg, 'dgcnn'); plt.show()
T.plot_confusion(res_dg, model_name='dgcnn'); plt.show()

## 2. DHSLP — ipergrafo learned (fedele a EEG_13b) su segnale grezzo

In [ ]:
df_hg, res_hg = T.run_subject_dependent(
    'dhslp', model_kwargs=dict(K=4, n_edges=16, d_model=64, hidden=64),
    train_kwargs=dict(epochs=200, patience=30, lr=1e-3, batch_size=32))
T.save_metrics(df_hg, 'dhslp')
T.plot_per_subject(df_hg, 'dhslp'); plt.show()
T.plot_confusion(res_hg, model_name='dhslp'); plt.show()

## 3. Curve di training di un soggetto (esempio DHSLP)

In [ ]:
T.plot_training_curves(res_hg, subject=1); plt.show()

### Nota sui modelli a grafo
- **DGCNN** ora costruisce un **grafo PCC/PLV per-trial + pruning top-k** (tecnica base della tesi,
  in `track3_graphs.py`), con node features dal segnale grezzo — non più band-power.
- **DHSLP** è la versione **fedele a EEG_13b** (ipergrafo con iperarchi appresi, incidenza soft).
  Per la pipeline gamma/semi-supervised di Li et al. vedi EEG_15 (`project_beat_liet_al` in memoria).

Iperparametri utili: DGCNN `metric='pcc'|'plv'`, `k_neighbors` (pruning); DHSLP `n_edges`, `K`.